# CT Corruption Detection - New Augmentation + Architecture Benchmarking

**Purpose:** Benchmark 3 new corruption types + combo mode across 3 architectures (ResNet-18, U-Net, EfficientNet).
Preprocessing code reused from Gnanavel's ct_corruption_medicalnet.ipynb for fair comparison.

### New Corruption Types:
- **Anatomical Slicing** - 5 sub-modes (top, bottom, both, middle-gap, scattered)
- **Gaussian Noise** - electronic interference / aging equipment
- **Resolution Downsampling** - low-cost scanner / lossy compression

### Combo Modes: single (1 type), double (2 types), triple (all 3)

### Architectures Benchmarked:
- **3D ResNet-18** (MedicalNet pretrained)
- **3D U-Net** (MONAI)
- **3D EfficientNet** (MONAI)

## 1) Imports & Setup

In [ ]:
# ===========================================================================
# Based on Gnanavel's implementation in ct_corruption_medicalnet.ipynb
# Preprocessing code reused for experiment consistency.
# New contributions by Dou Gwon:
#   - 3 new corruption functions (anatomical_slicing, gaussian_noise, resolution_downsampling)
#   - Combo corruption logic (single/double/triple)
#   - 3-architecture benchmarking (ResNet-18, U-Net, EfficientNet)
# ===========================================================================

import time
NOTEBOOK_START = time.time()

# --- Imports (reused from Gnanavel's notebook) ---
import os, glob, math, random, shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Medical image IO
import SimpleITK as sitk

# Resize + blur helpers
from scipy.ndimage import zoom, gaussian_filter

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

# Evaluation metrics
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_recall_fscore_support, roc_auc_score,
    ConfusionMatrixDisplay
)

# --- MONAI (new dependency for U-Net and EfficientNet) ---
!pip -q install monai
import monai

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"MONAI:   {monai.__version__}")
print("All imports done.")

## 2) Configuration

In [ ]:
# â”€â”€ Configuration â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

OUT_DIR = Path("./dataset_npz_new_aug")
TARGET_SHAPE = (64, 128, 128)    # (D, H, W) after resize
HU_MIN, HU_MAX = -1000, 400     # lung window

# New corruption parameters (Dou Gwon)
SLICE_REMOVAL_FRAC = (0.15, 0.35)
SLICING_MODES = ("top", "bottom", "both", "middle", "scattered")
NOISE_STD_RANGE = (30.0, 80.0)
DOWNSCALE_FACTOR_RANGE = (0.3, 0.6)
COMBO_WEIGHTS = {"single": 0.60, "double": 0.25, "triple": 0.15}

# Data splits
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

# Training
BATCH_SIZE = 8
PHASE1_EPOCHS = 30
PHASE2_EPOCHS = 30
PHASE1_LR = 3e-4
PHASE2_LR = 1e-5

# Models to benchmark
MODEL_NAMES = ["resnet18", "unet", "efficientnet"]

# Kaggle paths
BASE_DIR = "/kaggle/input/datasets/fardinabdullaacanto"
MEDICALNET_CKPT = os.path.join(BASE_DIR, "medicalnet-weights/resnet_18_23dataset.pth")

RANDOM_SEED = 42

# Print all config
print("=" * 50)
print("CONFIGURATION")
print("=" * 50)
print(f"  Output dir:          {OUT_DIR}")
print(f"  Target shape:        {TARGET_SHAPE}")
print(f"  HU window:           [{HU_MIN}, {HU_MAX}]")
print(f"  Slice removal frac:  {SLICE_REMOVAL_FRAC}")
print(f"  Slicing modes:       {SLICING_MODES}")
print(f"  Noise std range:     {NOISE_STD_RANGE}")
print(f"  Downscale factor:    {DOWNSCALE_FACTOR_RANGE}")
print(f"  Combo weights:       {COMBO_WEIGHTS}")
print(f"  Data split:          {TRAIN_FRAC}/{VAL_FRAC}/{TEST_FRAC}")
print(f"  Batch size:          {BATCH_SIZE}")
print(f"  Training:            {PHASE1_EPOCHS}+{PHASE2_EPOCHS} = {PHASE1_EPOCHS+PHASE2_EPOCHS} epochs")
print(f"  Phase 1 LR:          {PHASE1_LR}")
print(f"  Phase 2 LR:          {PHASE2_LR}")
print(f"  Models:              {MODEL_NAMES}")
print(f"  Random seed:         {RANDOM_SEED}")
print("=" * 50)

## 3) Data Discovery â€” LUNA16 Subsets

In [ ]:
# Reused from Gnanavel's notebook â€” auto-discover all 10 LUNA16 subsets

all_mhd_paths = []
subset_counts = {}

for i in range(10):
    # Try common path patterns for each subset
    candidates = [
        os.path.join(BASE_DIR, f"subset{i}", f"subset{i}"),
        os.path.join(BASE_DIR, f"subset{i}", f"subset{i}", f"subset{i}"),
        os.path.join(BASE_DIR, f"subset{i}"),
    ]

    found = False
    for path in candidates:
        mhds = sorted(glob.glob(os.path.join(path, "*.mhd")))
        if len(mhds) > 0:
            all_mhd_paths.extend(mhds)
            subset_counts[f"subset{i}"] = len(mhds)
            print(f"  subset{i}: {len(mhds):3d} scans  ({path})")
            found = True
            break

    if not found:
        # Try recursive search
        mhds = sorted(glob.glob(os.path.join(BASE_DIR, f"subset{i}", "**", "*.mhd"), recursive=True))
        if len(mhds) > 0:
            all_mhd_paths.extend(mhds)
            subset_counts[f"subset{i}"] = len(mhds)
            print(f"  subset{i}: {len(mhds):3d} scans  (recursive)")
        else:
            print(f"  subset{i}:   0 scans  (NOT FOUND)")

print(f"\nTOTAL: {len(all_mhd_paths)} scans across {len(subset_counts)} subsets")

## 4) Preprocessing Functions

In [ ]:
# Reused from Gnanavel's notebook â€” identical preprocessing for experiment consistency

def load_mhd(path: str) -> np.ndarray:
    """Load .mhd/.raw CT volume. Returns (D, H, W) float32 in HU."""
    img = sitk.ReadImage(path)
    arr = sitk.GetArrayFromImage(img)
    return arr.astype(np.float32)

def clip_hu(vol: np.ndarray, lo: int = -1000, hi: int = 400) -> np.ndarray:
    """Clip volume to Hounsfield Unit lung window."""
    return np.clip(vol, lo, hi)

def normalize_01(vol: np.ndarray, lo: int = -1000, hi: int = 400) -> np.ndarray:
    """Normalize clipped volume to [0, 1] range."""
    return ((vol - lo) / (hi - lo)).astype(np.float32)

def resize_dhw(vol: np.ndarray, out_shape: tuple = (64, 128, 128)) -> np.ndarray:
    """Resize volume to target (D, H, W) using trilinear interpolation."""
    D, H, W = vol.shape
    d2, h2, w2 = out_shape
    return zoom(vol, (d2 / D, h2 / H, w2 / W), order=1).astype(np.float32)

def preprocess(vol_hu: np.ndarray, out_shape: tuple = TARGET_SHAPE) -> np.ndarray:
    """Full pipeline: clip -> normalize -> resize."""
    vol = clip_hu(vol_hu, HU_MIN, HU_MAX)
    vol = normalize_01(vol, HU_MIN, HU_MAX)
    vol = resize_dhw(vol, out_shape)
    return vol

# Test: load first and middle volumes, print shapes
vol0 = load_mhd(all_mhd_paths[0])
print(f"Volume 0 shape: {vol0.shape}, HU range: [{vol0.min():.0f}, {vol0.max():.0f}]")

vol_mid = load_mhd(all_mhd_paths[len(all_mhd_paths) // 2])
print(f"Volume mid shape: {vol_mid.shape}, HU range: [{vol_mid.min():.0f}, {vol_mid.max():.0f}]")

vol_p = preprocess(vol0)
print(f"Preprocessed: {vol_p.shape}, range [{vol_p.min():.2f}, {vol_p.max():.2f}]")

## 5) Visualization Helpers

In [ ]:
# Reused from Gnanavel's notebook

def show_slice(vol: np.ndarray, idx: int, vmin: float = -1000, vmax: float = 400, title: str = None) -> None:
    """Display a single slice from a 3D volume."""
    plt.figure(figsize=(4, 4))
    plt.imshow(vol[idx], cmap="gray", vmin=vmin, vmax=vmax)
    plt.axis("off")
    plt.title(title or f"slice {idx}")
    plt.show()

def show_comparison(vol_a: np.ndarray, vol_b: np.ndarray, idx: int,
                    title_a: str = "A", title_b: str = "B") -> None:
    """Side-by-side comparison with absolute difference."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(vol_a[idx], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title(title_a); axes[0].axis("off")
    axes[1].imshow(vol_b[idx], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title(title_b); axes[1].axis("off")
    axes[2].imshow(np.abs(vol_a[idx] - vol_b[idx]), cmap="hot")
    axes[2].set_title("Absolute Difference"); axes[2].axis("off")
    plt.tight_layout()
    plt.show()

# Quick test: show middle slice of first loaded volume
mid = vol0.shape[0] // 2
show_slice(vol0, mid, title="Original CT (lung window)")

## 6) New Corruption Functions

In [ ]:
def apply_anatomical_slicing(vol_hu: np.ndarray, removal_frac: float,
                             mode: str = "random",
                             rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Simulate anatomical incompleteness by removing slices from a 3D CT volume.

    This corruption models real-world scenarios where CT scans are incomplete
    due to equipment, operator, or data-handling failures common in low-resource
    medical settings.

    Five sub-modes, each targeting a different failure scenario:

    1. "top" â€” Remove slices from the top (superior) end of the volume.
       Scenario: Late scan start due to patient positioning error. The
       technician begins acquisition after the scanner has already passed
       the top of the intended anatomy.

    2. "bottom" â€” Remove slices from the bottom (inferior) end of the volume.
       Scenario: Early scan termination caused by equipment malfunction,
       patient discomfort, or time pressure in a busy facility with limited
       scanner availability.

    3. "both" â€” Remove slices from both the top and bottom ends.
       Scenario: Incorrectly configured scan range where the technician
       sets both the start and end positions too narrowly, missing anatomy
       at both extremes.

    4. "middle" â€” Remove a contiguous block of slices from the middle.
       Scenario: Scanner interruption and restart (e.g., temporary power
       failure mid-scan), data transfer block loss between acquisition
       console and storage, or DICOM segment corruption during archival.

    5. "scattered" â€” Remove random individual slices throughout the volume.
       Scenario: Storage media corruption (e.g., bad sectors on aging hard
       drives), network packet loss during DICOM transfer over unreliable
       connections, or partial file corruption during backup/restore.

    Parameters
    ----------
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    removal_frac : float
        Fraction of slices to remove, in [0, 1).
    mode : str
        One of "top", "bottom", "both", "middle", "scattered", or "random".
        If "random", one of the 5 modes is chosen uniformly at random.
    rng : np.random.Generator, optional
        NumPy random generator for reproducibility.

    Returns
    -------
    tuple[np.ndarray, dict]
        (truncated_volume, info_dict) where info_dict contains:
        - "mode": the slicing mode used
        - "slices_removed": number of slices removed
        - "original_D": original depth
        - "removal_frac_actual": actual fraction removed
    """
    rng = rng or np.random.default_rng()
    D, H, W = vol_hu.shape
    n_remove = max(1, int(D * removal_frac))
    n_remove = min(n_remove, D - 1)  # keep at least 1 slice

    if mode == "random":
        mode = rng.choice(["top", "bottom", "both", "middle", "scattered"])

    if mode == "top":
        result = vol_hu[n_remove:]

    elif mode == "bottom":
        result = vol_hu[:D - n_remove]

    elif mode == "both":
        n_top = n_remove // 2
        n_bot = n_remove - n_top
        result = vol_hu[n_top:D - n_bot]

    elif mode == "middle":
        # Place the gap within the central 50% of the volume
        earliest_start = int(D * 0.25)
        latest_start = max(earliest_start, int(D * 0.75) - n_remove)
        start = rng.integers(earliest_start, latest_start + 1)
        result = np.concatenate([vol_hu[:start], vol_hu[start + n_remove:]], axis=0)

    elif mode == "scattered":
        keep_mask = np.ones(D, dtype=bool)
        remove_indices = rng.choice(D, size=n_remove, replace=False)
        keep_mask[remove_indices] = False
        result = vol_hu[keep_mask]

    else:
        raise ValueError(f"Unknown slicing mode: {mode!r}. "
                         f"Expected one of: top, bottom, both, middle, scattered, random.")

    info = {
        "mode": mode,
        "slices_removed": n_remove,
        "original_D": D,
        "removal_frac_actual": round(n_remove / D, 4),
    }
    return result, info

In [ ]:
print("--- Testing apply_anatomical_slicing ---")
dummy = np.random.randn(200, 512, 512).astype(np.float32)
for mode in ["top", "bottom", "both", "middle", "scattered"]:
    result, info = apply_anatomical_slicing(dummy, removal_frac=0.25, mode=mode, rng=np.random.default_rng(42))
    assert result.ndim == 3
    assert result.shape[1] == 512 and result.shape[2] == 512
    assert result.shape[0] < dummy.shape[0]
    assert result.shape[0] > 0
    print(f"  {mode:10s}: {dummy.shape} -> {result.shape}, removed {info['slices_removed']} slices")
print("PASSED: apply_anatomical_slicing all 5 modes")

In [ ]:
def apply_gaussian_noise(vol_hu: np.ndarray, std: float,
                         rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Add Gaussian noise to a 3D CT volume to simulate electronic interference.

    This corruption models real-world scenarios where image quality is degraded
    by hardware limitations common in low-resource medical settings:

    - Aging or low-quality CT detector electronics producing elevated noise
      floors that obscure fine anatomical detail.
    - Power supply instability in under-resourced facilities causing signal
      fluctuations that manifest as random intensity variations across the
      reconstructed volume.
    - Sensor degradation in equipment that cannot be regularly maintained or
      calibrated, leading to progressively noisier acquisitions over time.

    Parameters
    ----------
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    std : float
        Standard deviation of the Gaussian noise in HU. Typical range:
        30-80 HU for realistic corruption.
    rng : np.random.Generator, optional
        NumPy random generator for reproducibility.

    Returns
    -------
    tuple[np.ndarray, dict]
        (noisy_volume, info_dict) where info_dict contains:
        - "std": the noise standard deviation used
    """
    rng = rng or np.random.default_rng()
    noise = rng.normal(0.0, std, size=vol_hu.shape).astype(np.float32)
    noisy = vol_hu + noise
    # Clip to 12-bit CT physical bounds
    noisy = np.clip(noisy, -1024.0, 3071.0)
    return noisy, {"std": float(std)}

In [ ]:
print("--- Testing apply_gaussian_noise ---")
dummy = np.full((100, 64, 64), -500.0, dtype=np.float32)
noisy, info = apply_gaussian_noise(dummy, std=50.0, rng=np.random.default_rng(42))
actual_noise = noisy - dummy
print(f"  noise mean: {actual_noise.mean():.2f} (expect ~0)")
print(f"  noise std:  {actual_noise.std():.2f} (expect ~50)")
print(f"  output range: [{noisy.min():.0f}, {noisy.max():.0f}]")
assert noisy.shape == dummy.shape
assert abs(actual_noise.mean()) < 1.0
assert abs(actual_noise.std() - 50.0) < 2.0
print("PASSED: apply_gaussian_noise")

In [ ]:
def apply_resolution_downsampling(vol_hu: np.ndarray, factor: float,
                                  rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Degrade spatial resolution of a 3D CT volume by downsampling then upsampling.

    This corruption models real-world scenarios where image resolution is lost
    due to hardware or infrastructure limitations in low-resource settings:

    - Low-cost or legacy CT scanners with inherently poor spatial resolution,
      common in under-funded hospitals that cannot afford modern equipment.
    - Lossy compression applied to save storage in resource-constrained
      facilities where disk space is limited and lossless archival is not
      feasible.
    - Bandwidth-limited remote transmission: volumes are downsampled before
      sending over slow network links (e.g., rural telemedicine), then
      reconstructed at the receiving end, permanently losing fine detail.

    The degradation is achieved by zooming the volume down to a smaller size
    (controlled by `factor`) and then zooming back up to the original size.
    This round-trip through a lower resolution irreversibly destroys
    high-frequency spatial information.

    Parameters
    ----------
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    factor : float
        Downscale factor in (0, 1). Smaller values produce more aggressive
        degradation. For example, factor=0.5 reduces each dimension to 50%
        before restoring. Typical range: 0.3-0.6.
    rng : np.random.Generator, optional
        NumPy random generator (unused, accepted for API consistency).

    Returns
    -------
    tuple[np.ndarray, dict]
        (restored_volume, info_dict) where info_dict contains:
        - "factor": the downscale factor used
        - "reduced_shape": the intermediate reduced shape

    Raises
    ------
    ValueError
        If factor is not in the open interval (0, 1).
    """
    if not (0 < factor < 1):
        raise ValueError(f"factor must be in (0, 1), got {factor}")

    orig_shape = vol_hu.shape
    reduced = tuple(max(1, int(s * factor)) for s in orig_shape)

    # Downscale to reduced resolution
    zoom_down = [r / o for r, o in zip(reduced, orig_shape)]
    small = zoom(vol_hu, zoom_down, order=1)

    # Upscale back to original resolution
    zoom_up = [o / s for o, s in zip(orig_shape, small.shape)]
    restored = zoom(small, zoom_up, order=1)

    # Fix any shape mismatch from rounding in zoom
    if restored.shape != orig_shape:
        slices = tuple(slice(0, s) for s in orig_shape)
        restored = restored[slices]

    return restored.astype(np.float32), {"factor": float(factor), "reduced_shape": reduced}

In [ ]:
print("--- Testing apply_resolution_downsampling ---")
rng = np.random.default_rng(42)
dummy = rng.normal(-500, 200, (100, 64, 64)).astype(np.float32)  # Random data with variation to show information loss
dummy = np.clip(dummy, -1024, 3071)  # Clip to valid HU range
degraded, info = apply_resolution_downsampling(dummy, factor=0.5)
mae = np.abs(degraded - dummy).mean()
print(f"  original:  {dummy.shape}")
print(f"  reduced:   {info['reduced_shape']}")
print(f"  restored:  {degraded.shape}")
print(f"  MAE:       {mae:.2f} (should be > 0, information lost)")
assert degraded.shape == dummy.shape
assert mae > 0, f"Expected MAE > 0, got {mae:.2f}. Try increasing variation in dummy data or reducing factor."
print("PASSED: apply_resolution_downsampling")

In [ ]:
print("=" * 60)
print("INTEGRATION TEST: All Corruption Functions")
print("=" * 60)

rng = np.random.default_rng(42)
dummy = rng.normal(-500, 200, size=(200, 512, 512)).astype(np.float32)
dummy = np.clip(dummy, -1024, 3071)
print(f"Dummy volume: shape={dummy.shape}, range=[{dummy.min():.0f}, {dummy.max():.0f}]")
print()

# 1. Anatomical slicing â€” all 5 modes
print("1) Anatomical Slicing:")
for mode in ["top", "bottom", "both", "middle", "scattered"]:
    r, info = apply_anatomical_slicing(dummy, 0.25, mode=mode, rng=np.random.default_rng(42))
    assert r.ndim == 3 and r.shape[1] == 512 and r.shape[0] < 200
    print(f"   {mode:10s}: {dummy.shape} -> {r.shape} (removed {info['slices_removed']})")

# 2. Gaussian noise
print("\n2) Gaussian Noise:")
noisy, info = apply_gaussian_noise(dummy, std=50.0, rng=np.random.default_rng(42))
assert noisy.shape == dummy.shape
print(f"   shape unchanged: {noisy.shape}, std applied: {info['std']}")

# 3. Resolution downsampling
print("\n3) Resolution Downsampling:")
deg, info = apply_resolution_downsampling(dummy, factor=0.5)
assert deg.shape == dummy.shape
print(f"   shape unchanged: {deg.shape}, factor: {info['factor']}, reduced: {info['reduced_shape']}")

# 4. Verify preprocessing still works after each corruption
print("\n4) Preprocessing after corruption:")
for name, vol in [("sliced", apply_anatomical_slicing(dummy, 0.25, "middle", rng=np.random.default_rng(42))[0]),
                   ("noisy", noisy),
                   ("degraded", deg)]:
    pp = preprocess(vol)
    assert pp.shape == tuple(TARGET_SHAPE), f"{name}: expected {TARGET_SHAPE}, got {pp.shape}"
    assert pp.min() >= 0.0 and pp.max() <= 1.0
    print(f"   {name:10s}: preprocessed to {pp.shape}, range [{pp.min():.3f}, {pp.max():.3f}]")

print("\n" + "=" * 60)
print("ALL INTEGRATION TESTS PASSED")
print("=" * 60)

## 7) Combo Corruption Logic

In [ ]:
def apply_combo_corruption(vol_hu: np.ndarray, corruption_list: list[str],
                           params_dict: dict,
                           rng: np.random.Generator = None) -> tuple[np.ndarray, dict]:
    """
    Apply multiple corruption types sequentially to a 3D CT volume.

    Combo corruption simulates real-world scenarios where CT scans suffer from
    multiple simultaneous quality issues â€” e.g., a noisy scan from aging
    equipment that also has missing slices due to a transfer error.

    Ordering rule: `anatomical_slicing` is ALWAYS applied first if present,
    because it changes the depth dimension. Gaussian noise and resolution
    downsampling preserve shape and can follow in any order.

    Parameters
    ----------
    vol_hu : np.ndarray
        3D CT volume in Hounsfield Units, shape (D, H, W).
    corruption_list : list[str]
        List of corruption type names to apply. Valid names:
        "anatomical_slicing", "gaussian_noise", "resolution_downsampling".
    params_dict : dict
        Keyword arguments for each corruption function, keyed by corruption
        name. Example::

            {
                "anatomical_slicing": {"removal_frac": 0.2, "mode": "middle"},
                "gaussian_noise": {"std": 50.0},
                "resolution_downsampling": {"factor": 0.5},
            }
    rng : np.random.Generator, optional
        NumPy random generator for reproducibility.

    Returns
    -------
    tuple[np.ndarray, dict]
        (corrupted_volume, combined_info) where combined_info contains:
        - "types_applied": ordered list of corruption types as applied
        - per-corruption info dicts keyed by corruption name
    """
    rng = rng or np.random.default_rng()

    # Sort: anatomical_slicing first (changes depth), then others alphabetically
    sorted_list = sorted(corruption_list,
                         key=lambda x: (0 if x == "anatomical_slicing" else 1, x))

    combined_info: dict = {"types_applied": sorted_list}

    dispatch = {
        "anatomical_slicing": apply_anatomical_slicing,
        "gaussian_noise": apply_gaussian_noise,
        "resolution_downsampling": apply_resolution_downsampling,
    }

    for ctype in sorted_list:
        fn = dispatch[ctype]
        kwargs = params_dict.get(ctype, {})
        vol_hu, info = fn(vol_hu, **kwargs, rng=rng)
        combined_info[ctype] = info

    return vol_hu, combined_info

In [ ]:
print("--- Testing Combo Corruption ---")
rng = np.random.default_rng(42)
dummy = rng.normal(-500, 200, (200, 512, 512)).astype(np.float32)

# Test combo_2: slicing + noise
params_2 = {
    "anatomical_slicing": {"removal_frac": 0.2, "mode": "middle"},
    "gaussian_noise": {"std": 50.0}
}
r2, info2 = apply_combo_corruption(dummy, ["gaussian_noise", "anatomical_slicing"], params_2, rng=np.random.default_rng(42))
assert info2["types_applied"][0] == "anatomical_slicing", "slicing must be first!"
assert r2.shape[0] < 200 and r2.shape[1] == 512
print(f"  combo_2: {dummy.shape} -> {r2.shape}, applied: {info2['types_applied']}")

# Test combo_3: all 3
params_3 = {
    "anatomical_slicing": {"removal_frac": 0.2, "mode": "scattered"},
    "gaussian_noise": {"std": 60.0},
    "resolution_downsampling": {"factor": 0.5}
}
r3, info3 = apply_combo_corruption(dummy, ["resolution_downsampling", "gaussian_noise", "anatomical_slicing"], params_3, rng=np.random.default_rng(42))
assert info3["types_applied"][0] == "anatomical_slicing"
assert len(info3["types_applied"]) == 3
print(f"  combo_3: {dummy.shape} -> {r3.shape}, applied: {info3['types_applied']}")

# Verify preprocessing works on combo results
pp = preprocess(r3)
assert pp.shape == tuple(TARGET_SHAPE)
print(f"  combo_3 preprocessed: {pp.shape}")

print("PASSED: Combo corruption logic")

## 8) Dataset Builder

In [ ]:
def pick_scan_label(D_original: int, mode: str = "random", p_corrupt: float = 0.5,
                    rng: np.random.Generator = None) -> tuple[int, dict]:
    """
    Decide whether a scan is clean or corrupted, and generate corruption params.

    For corrupted samples, uses COMBO_WEIGHTS to decide the combo level:
    - 60% single (1 of 3 types)
    - 25% combo_2 (2 of 3 types)
    - 15% combo_3 (all 3 types)

    Then generates random parameters for each selected corruption type within
    the configured ranges.

    Parameters
    ----------
    D_original : int
        Original depth (number of slices) of the volume â€” used for context.
    mode : str
        "random" for stochastic labeling, "clean" to force clean, "corrupt"
        to force corrupted.
    p_corrupt : float
        Probability of assigning a corrupt label (default 0.5 for balance).
    rng : np.random.Generator, optional
        NumPy random generator for reproducibility.

    Returns
    -------
    tuple[int, dict]
        (label, meta) where label is 0 (clean) or 1 (corrupted), and meta
        contains "type" (str) and corruption parameters if corrupted.
    """
    rng = rng or np.random.default_rng()

    if mode == "clean":
        return 0, {"type": "clean"}
    if mode == "corrupt":
        is_corrupt = True
    else:
        is_corrupt = rng.random() < p_corrupt

    if not is_corrupt:
        return 0, {"type": "clean"}

    # Decide combo level using COMBO_WEIGHTS
    combo_keys = list(COMBO_WEIGHTS.keys())
    combo_probs = np.array([COMBO_WEIGHTS[k] for k in combo_keys])
    combo_probs = combo_probs / combo_probs.sum()
    combo_level = rng.choice(combo_keys, p=combo_probs)

    if combo_level == "single":
        n_types = 1
    elif combo_level == "double":
        n_types = 2
    else:  # triple
        n_types = 3

    # Pick which corruption types to apply
    chosen_types = list(rng.choice(ALL_CORRUPTION_TYPES, size=n_types, replace=False))

    # Generate random parameters for each chosen type
    params = {}
    for ctype in chosen_types:
        if ctype == "anatomical_slicing":
            frac = rng.uniform(*SLICE_REMOVAL_FRAC)
            slicing_mode = rng.choice(SLICING_MODES)
            params[ctype] = {"removal_frac": float(frac), "mode": str(slicing_mode)}
        elif ctype == "gaussian_noise":
            std = rng.uniform(*NOISE_STD_RANGE)
            params[ctype] = {"std": float(std)}
        elif ctype == "resolution_downsampling":
            factor = rng.uniform(*DOWNSCALE_FACTOR_RANGE)
            params[ctype] = {"factor": float(factor)}

    # Determine type label
    if n_types == 1:
        ctype_label = chosen_types[0]
    elif n_types == 2:
        ctype_label = "combo_2"
    else:
        ctype_label = "combo_3"

    meta = {
        "type": ctype_label,
        "corruption_types": chosen_types,
        "params": params,
    }
    return 1, meta


# Adapted from Gnanavel's corrupt_and_preprocess() â€” routes to new corruption functions

def corrupt_and_preprocess(vol_hu: np.ndarray, meta: dict,
                           rng: np.random.Generator = None) -> np.ndarray:
    """
    Apply corruption (if any) then preprocess a volume to model-ready format.

    Routes based on meta["type"]:
    - "clean" -> preprocess only
    - single type -> apply that corruption then preprocess
    - "combo_2" / "combo_3" -> apply_combo_corruption then preprocess

    Corruption is applied at full resolution BEFORE preprocessing (see D-001).

    Parameters
    ----------
    vol_hu : np.ndarray
        Raw 3D CT volume in Hounsfield Units, shape (D, H, W).
    meta : dict
        Metadata from pick_scan_label(), containing "type", "corruption_types",
        and "params".
    rng : np.random.Generator, optional
        NumPy random generator for reproducibility.

    Returns
    -------
    np.ndarray
        Preprocessed volume, shape TARGET_SHAPE, range [0, 1].
    """
    rng = rng or np.random.default_rng()
    ctype = meta["type"]

    if ctype == "clean":
        return preprocess(vol_hu)

    corruption_types = meta["corruption_types"]
    params = meta["params"]

    if ctype in ("combo_2", "combo_3"):
        vol_hu, _ = apply_combo_corruption(vol_hu, corruption_types, params, rng=rng)
    else:
        # Single corruption type
        single_type = corruption_types[0]
        dispatch = {
            "anatomical_slicing": apply_anatomical_slicing,
            "gaussian_noise": apply_gaussian_noise,
            "resolution_downsampling": apply_resolution_downsampling,
        }
        fn = dispatch[single_type]
        vol_hu, _ = fn(vol_hu, **params[single_type], rng=rng)

    return preprocess(vol_hu)


# Adapted from Gnanavel's make_dataset() â€” updated for new corruption types

def make_dataset(mhd_list: list[str], split: str = "train", seed: int = 0) -> dict:
    """
    Build .npz dataset from raw .mhd scans with 50/50 clean/corrupt balance.

    For each scan path, decides clean vs corrupt via pick_scan_label(), applies
    corruption at full resolution, preprocesses, and saves as .npz with keys
    "vol", "label", "ctype".

    Parameters
    ----------
    mhd_list : list[str]
        List of .mhd file paths.
    split : str
        Split name ("train", "val", or "test"). Used for output subdirectory.
    seed : int
        Random seed for this split.

    Returns
    -------
    dict
        Statistics dict with counts per corruption type.
    """
    rng = np.random.default_rng(seed)
    out_path = OUT_DIR / split
    out_path.mkdir(parents=True, exist_ok=True)

    stats = {
        "clean": 0, "anatomical_slicing": 0, "gaussian_noise": 0,
        "resolution_downsampling": 0, "combo_2": 0, "combo_3": 0,
    }
    n_total = len(mhd_list)

    for i, mhd_path in enumerate(mhd_list):
        try:
            vol_hu = load_mhd(mhd_path)
            D_original = vol_hu.shape[0]
            label, meta = pick_scan_label(D_original, mode="random", rng=rng)
            vol_pp = corrupt_and_preprocess(vol_hu, meta, rng=rng)

            ctype = meta["type"]
            stats[ctype] = stats.get(ctype, 0) + 1

            fname = f"{split}_{i:04d}.npz"
            np.savez_compressed(
                str(out_path / fname),
                vol=vol_pp,
                label=np.int64(label),
                ctype=ctype,
            )

            if (i + 1) % 50 == 0 or (i + 1) == n_total:
                print(f"  [{split}] {i+1}/{n_total} processed")

        except Exception as e:
            print(f"  [{split}] ERROR on {mhd_path}: {e}")

    print(f"  [{split}] Done: {n_total} scans -> {out_path}")
    print(f"  [{split}] Stats: {stats}")
    return stats

In [ ]:
print("--- Testing Dataset Builder ---")
rng = np.random.default_rng(42)
stats = {"clean": 0, "anatomical_slicing": 0, "gaussian_noise": 0,
         "resolution_downsampling": 0, "combo_2": 0, "combo_3": 0}

for _ in range(100):
    label, meta = pick_scan_label(200, mode="random", rng=rng)
    if meta["type"] in stats:
        stats[meta["type"]] += 1

print("Distribution over 100 samples:")
for k, v in stats.items():
    print(f"  {k:25s}: {v:3d} ({v}%)")

clean_pct = stats["clean"]
corrupt_pct = 100 - clean_pct
print(f"\nClean: ~{clean_pct}%, Corrupt: ~{corrupt_pct}% (expect ~50/50)")
assert 35 <= clean_pct <= 65, f"Balance off: {clean_pct}% clean"

# Verify all 6 types appear
all_types_present = all(v > 0 for v in stats.values())
if not all_types_present:
    missing = [k for k, v in stats.items() if v == 0]
    print(f"  WARNING: missing types (may occur with small N=100): {missing}")
    print("  Re-running with N=1000 for better coverage...")
    rng2 = np.random.default_rng(123)
    stats2 = {"clean": 0, "anatomical_slicing": 0, "gaussian_noise": 0,
              "resolution_downsampling": 0, "combo_2": 0, "combo_3": 0}
    for _ in range(1000):
        label, meta = pick_scan_label(200, mode="random", rng=rng2)
        if meta["type"] in stats2:
            stats2[meta["type"]] += 1
    for k, v in stats2.items():
        print(f"  {k:25s}: {v:4d} ({v/10:.1f}%)")
    assert all(v > 0 for v in stats2.values()), f"Still missing types: {[k for k,v in stats2.items() if v==0]}"

print("PASSED: Dataset builder distribution")

## 9) Build Dataset (3-way split)

In [ ]:
# Dataset split logic from Gnanavel's notebook — same seed for reproducibility

import shutil

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
for split in ["train", "val", "test"]:
    (OUT_DIR / split).mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
paths = all_mhd_paths.copy()
random.shuffle(paths)

n_total = len(paths)
n_train = int(n_total * TRAIN_FRAC)
n_val = int(n_total * VAL_FRAC)
n_test = n_total - n_train - n_val

train_paths = paths[:n_train]
val_paths = paths[n_train:n_train + n_val]
test_paths = paths[n_train + n_val:]

print("=" * 50)
print("DATASET SPLIT")
print("=" * 50)
print(f"  Total scans:  {n_total}")
print(f"  Train:        {len(train_paths)} ({100*len(train_paths)/n_total:.1f}%)")
print(f"  Validation:   {len(val_paths)} ({100*len(val_paths)/n_total:.1f}%)")
print(f"  Test:         {len(test_paths)} ({100*len(test_paths)/n_total:.1f}%)")
print("=" * 50)

# Build each split
print("\nBuilding train set...")
train_stats = make_dataset(train_paths, split="train", seed=RANDOM_SEED)

print("\nBuilding validation set...")
val_stats = make_dataset(val_paths, split="val", seed=RANDOM_SEED + 1)

print("\nBuilding test set...")
test_stats = make_dataset(test_paths, split="test", seed=RANDOM_SEED + 2)

## 10) Dataset Statistics & Sample Visualization

In [ ]:
# Reused from Gnanavel's notebook — reads .npz files and counts labels/ctypes

def label_stats(folder) -> dict:
    """
    Compute label and corruption-type statistics for a dataset split.

    Parameters
    ----------
    folder : Path or str
        Directory containing .npz files with "label" and "ctype" keys.

    Returns
    -------
    dict
        Dictionary with keys: "N" (total), "clean" (count), "corrupt" (count),
        "types" (dict of ctype -> count).
    """
    labels, ctypes = [], []
    for p in Path(folder).glob("*.npz"):
        d = np.load(p, allow_pickle=True)
        labels.append(int(d["label"]))
        ctypes.append(str(d["ctype"]))
    labels = np.array(labels)
    counts = {}
    for ct in ctypes:
        counts[ct] = counts.get(ct, 0) + 1
    return {"N": len(labels), "clean": int((labels == 0).sum()),
            "corrupt": int((labels == 1).sum()), "types": counts}


# ── Print stats for all splits ──
print("=" * 60)
print("DATASET STATISTICS")
print("=" * 60)
for split in ["train", "val", "test"]:
    s = label_stats(OUT_DIR / split)
    print(f"\n  {split.upper():5s}: {s['N']} total | {s['clean']} clean | {s['corrupt']} corrupt")
    for ctype, count in sorted(s["types"].items()):
        print(f"         {ctype:25s}: {count}")
print("=" * 60)

# ── Sample Visualization: 8 random training samples in 2x4 grid ──
np.random.seed(RANDOM_SEED)
train_npz_files = sorted(list((OUT_DIR / "train").glob("*.npz")))
sample_indices = np.random.choice(len(train_npz_files), size=min(8, len(train_npz_files)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Random Training Samples — Green=Clean, Red=Corrupted", fontsize=14)

for ax_idx, (ax, file_idx) in enumerate(zip(axes.flat, sample_indices)):
    data = np.load(train_npz_files[file_idx], allow_pickle=True)
    vol = data["vol"]
    label = int(data["label"])
    ctype = str(data["ctype"])

    # Show middle slice
    mid = vol.shape[0] // 2
    ax.imshow(vol[mid], cmap="gray", vmin=0, vmax=1)
    color = "red" if label == 1 else "green"
    title = f"{'CORRUPT' if label == 1 else 'CLEAN'}\n{ctype}"
    ax.set_title(title, color=color, fontsize=10, fontweight="bold")
    ax.axis("off")

# Hide unused axes if fewer than 8 samples
for ax in axes.flat[len(sample_indices):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

# ── Verification Cell ──
print("\n--- Dataset Build Verification ---")
for split in ["train", "val", "test"]:
    s = label_stats(OUT_DIR / split)
    assert s["N"] > 0, f"{split} is empty!"
    assert s["clean"] > 0, f"{split} has no clean samples"
    assert s["corrupt"] > 0, f"{split} has no corrupt samples"
    print(f"  {split}: {s['N']} samples, {s['types']}")
print("PASSED: Dataset build verified")

## 11) PyTorch Dataset & DataLoaders

In [ ]:
# Reused from Gnanavel's notebook without modification

class NPZScanDataset(Dataset):
    """PyTorch Dataset that loads preprocessed .npz CT volumes."""

    def __init__(self, folder):
        self.paths = sorted(Path(folder).glob("*.npz"))
        if len(self.paths) == 0:
            raise FileNotFoundError(f"No .npz in {folder}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        d = np.load(self.paths[idx], allow_pickle=True)
        vol = d["vol"].astype(np.float32)
        label = int(d["label"])
        vol = np.expand_dims(vol, axis=0)  # (1, D, H, W)
        return torch.from_numpy(vol), torch.tensor(label, dtype=torch.float32)


# Create datasets using OUT_DIR (new augmentation output)
train_ds = NPZScanDataset(OUT_DIR / "train")
val_ds   = NPZScanDataset(OUT_DIR / "val")
test_ds  = NPZScanDataset(OUT_DIR / "test")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# === TEST ===
print("--- DataLoader Test ---")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
x, y = next(iter(train_loader))
print(f"Batch: x={tuple(x.shape)}, y={tuple(y.shape)}")
assert x.shape == (BATCH_SIZE, 1, 64, 128, 128) or x.shape[0] <= BATCH_SIZE
assert y.shape[0] == x.shape[0]
print("PASSED: DataLoader works")

## 12) Model Definitions â€” 3 Architectures

In [ ]:
# ============================================================
# Model 1: MedicalNet ResNet-18
# Model from Gnanavel's ct_corruption_medicalnet.ipynb — reused for fair comparison
# ============================================================

!git clone https://github.com/Tencent/MedicalNet.git

import sys
sys.path.append("/kaggle/working/MedicalNet")

from models import resnet


class MedicalNetResNet18Binary(nn.Module):
    """
    3D ResNet-18 for binary classification of CT volume completeness.

    Uses Tencent MedicalNet backbone pretrained on 23 medical imaging datasets.
    Classification head: AdaptiveAvgPool3d(1) → Dropout(0.5) → Linear(512, 1).

    Source: Reused from Gnanavel's ct_corruption_medicalnet.ipynb for fair comparison.
    """

    def __init__(self) -> None:
        super().__init__()
        self.backbone = resnet.resnet18(
            sample_input_D=TARGET_SHAPE[0],
            sample_input_H=TARGET_SHAPE[1],
            sample_input_W=TARGET_SHAPE[2],
            num_seg_classes=2,
            shortcut_type='A',
            no_cuda=False
        )
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(512, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        x = self.backbone.layer4(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x).squeeze(1)


def load_medicalnet_weights(model: nn.Module, ckpt_path: str) -> None:
    """
    Load MedicalNet pretrained weights into the ResNet-18 backbone.

    Handles key renaming (module. prefix removal, backbone. prefix addition)
    and shape mismatch filtering. Prints summary of loaded/missing keys.

    Source: Reused from Gnanavel's ct_corruption_medicalnet.ipynb.

    Parameters
    ----------
    model : nn.Module
        MedicalNetResNet18Binary instance.
    ckpt_path : str
        Path to resnet_18_23dataset.pth checkpoint file.
    """
    if not os.path.isfile(ckpt_path):
        print(f"WARNING: {ckpt_path} not found. Training from scratch.")
        return
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = ckpt.get("state_dict", ckpt)
    model_sd = model.state_dict()
    new_sd = {}
    for k, v in sd.items():
        k = k.replace("module.", "")
        if not k.startswith("backbone."):
            k = "backbone." + k
        if k in model_sd and v.shape == model_sd[k].shape:
            new_sd[k] = v
    missing, _ = model.load_state_dict(new_sd, strict=False)
    print(f"Loaded {len(new_sd)}/{len(model_sd)} tensors from MedicalNet checkpoint.")
    print(f"Missing (seg head + fc): {len(missing)} keys")


# ============================================================
# Model 2: 3D U-Net (MONAI)
# New contribution by Dou Gwon — adapted for binary classification
# ============================================================

from monai.networks.nets import UNet


class UNet3DBinary(nn.Module):
    """
    3D U-Net for binary classification of CT volume completeness.

    Originally designed for segmentation, adapted for classification by
    running the full U-Net encoder-decoder and then applying global average
    pooling followed by a linear classification head.

    Architecture uses MONAI's UNet with channels (32, 64, 128, 256),
    followed by AdaptiveAvgPool3d and a binary classification head.

    Real-world relevance: U-Net's multi-scale feature extraction via skip
    connections makes it effective at detecting both local (noise, blur)
    and global (missing slices) corruption patterns. Its relatively small
    parameter count (~3-5M) makes it suitable for deployment in
    resource-constrained medical facilities.

    Parameters
    ----------
    None — uses fixed architecture configuration.

    Returns
    -------
    forward(x) returns tensor of shape (B,) with raw logits.
    """

    def __init__(self) -> None:
        super().__init__()
        self.backbone = UNet(
            spatial_dims=3,
            in_channels=1,
            out_channels=1,
            channels=(32, 64, 128, 256),
            strides=(2, 2, 2),
            num_res_units=2
        )
        # Classification head applied on top of U-Net output
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(1, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Run full U-Net (encoder + decoder with skip connections)
        # Output is a segmentation-like map of shape (B, 1, D, H, W)
        feat = self.backbone(x)
        # Global average pool to (B, 1, 1, 1, 1) then flatten to (B, 1)
        pooled = self.pool(feat).flatten(1)
        pooled = self.dropout(pooled)
        return self.fc(pooled).squeeze(1)


# ============================================================
# Model 3: 3D EfficientNet (MONAI)
# New contribution by Dou Gwon — lightweight model for low-resource deployment
# ============================================================

from monai.networks.nets import EfficientNetBN


class EfficientNet3DBinary(nn.Module):
    """
    3D EfficientNet for binary classification of CT volume completeness.

    Uses MONAI's EfficientNetBN with compound scaling for efficient
    parameter usage. Lightweight architecture suitable for deployment
    in low-resource medical settings.

    Real-world relevance: Smaller model size and lower inference time
    make this suitable for facilities with limited compute resources.
    EfficientNet's compound scaling balances depth, width, and resolution
    for optimal accuracy-per-parameter.

    Parameters
    ----------
    None — uses EfficientNet-B0 configuration with 3D spatial dims.

    Returns
    -------
    forward(x) returns tensor of shape (B,) with raw logits.
    """

    def __init__(self) -> None:
        super().__init__()
        self.backbone = EfficientNetBN(
            model_name="efficientnet-b0",
            spatial_dims=3,
            in_channels=1,
            num_classes=1  # binary classification logit
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x).squeeze(1)  # (B,) logits


print("All 3 model classes defined:")
print("  1. MedicalNetResNet18Binary  — MedicalNet pretrained ResNet-18")
print("  2. UNet3DBinary              — MONAI 3D U-Net adapted for classification")
print("  3. EfficientNet3DBinary      — MONAI 3D EfficientNet-B0")


## 13) Model Factory & Verification

In [ ]:
def create_model(name: str) -> nn.Module:
    """
    Factory function to create model by name.

    Parameters
    ----------
    name : str
        One of "resnet18", "unet", "efficientnet".

    Returns
    -------
    nn.Module
        Instantiated model. For resnet18, MedicalNet weights are loaded.

    Raises
    ------
    ValueError
        If name is not recognized.
    """
    if name == "resnet18":
        model = MedicalNetResNet18Binary()
        load_medicalnet_weights(model, MEDICALNET_CKPT)
        return model
    elif name == "unet":
        return UNet3DBinary()
    elif name == "efficientnet":
        return EfficientNet3DBinary()
    else:
        raise ValueError(f"Unknown model: {name}")


def count_parameters(model: nn.Module) -> int:
    """Return total number of parameters in the model."""
    return sum(p.numel() for p in model.parameters())


def count_trainable(model: nn.Module) -> int:
    """Return number of trainable (requires_grad=True) parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ============================================================
# CRITICAL TEST — Forward pass verification for all 3 models
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"
dummy_input = torch.randn(2, 1, 64, 128, 128).to(device)

print("=" * 70)
print("MODEL VERIFICATION — Forward Pass Test")
print("=" * 70)

for name in MODEL_NAMES:
    print(f"\n--- {name} ---")
    model = create_model(name).to(device)

    # Forward pass
    with torch.no_grad():
        output = model(dummy_input)

    print(f"  Input:      {tuple(dummy_input.shape)}")
    print(f"  Output:     {tuple(output.shape)}")
    print(f"  Parameters: {count_parameters(model):,}")

    # Verify output shape
    assert output.shape == (2,), f"{name}: expected output (2,), got {output.shape}"

    # Verify output is finite
    assert torch.isfinite(output).all(), f"{name}: output contains NaN/Inf"

    # Verify sigmoid produces valid probabilities
    probs = torch.sigmoid(output)
    assert (probs >= 0).all() and (probs <= 1).all()

    print(f"  Sigmoid:    [{probs.min().item():.4f}, {probs.max().item():.4f}]")
    print(f"  STATUS:     PASSED")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("ALL 3 MODELS VERIFIED — Forward pass, output shape, finite values")
print("=" * 70)

## 14) Training Functions (Model-Agnostic)

In [ ]:
# Training loop adapted from Gnanavel's ct_corruption_medicalnet.ipynb
# Made model-agnostic — accepts any nn.Module for fair architecture comparison


def train_one_epoch(model: nn.Module, loader: DataLoader,
                    optimizer: torch.optim.Optimizer,
                    criterion: nn.Module) -> tuple[float, float]:
    """
    Train model for one epoch.

    Parameters
    ----------
    model : nn.Module
        Model to train.
    loader : DataLoader
        Training data loader.
    optimizer : torch.optim.Optimizer
        Optimizer instance.
    criterion : nn.Module
        Loss function (e.g. BCEWithLogitsLoss).

    Returns
    -------
    tuple[float, float]
        (average_loss, accuracy) for this epoch.

    Source: Reused from Gnanavel's ct_corruption_medicalnet.ipynb.
    """
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x).view(-1)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == y).sum().item()
        n += x.size(0)
    return total_loss / max(1, n), correct / max(1, n)


@torch.no_grad()
def eval_one_epoch(model: nn.Module, loader: DataLoader,
                   criterion: nn.Module) -> tuple[float, float, np.ndarray, np.ndarray]:
    """
    Evaluate model for one epoch.

    Parameters
    ----------
    model : nn.Module
        Model to evaluate.
    loader : DataLoader
        Validation or test data loader.
    criterion : nn.Module
        Loss function.

    Returns
    -------
    tuple[float, float, np.ndarray, np.ndarray]
        (average_loss, accuracy, all_probs, all_labels).

    Source: Reused from Gnanavel's ct_corruption_medicalnet.ipynb.
    """
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    all_probs, all_labels = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x).view(-1)
        loss = criterion(logits, y)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        total_loss += loss.item() * x.size(0)
        correct += (preds == y).sum().item()
        n += x.size(0)
        all_probs.extend(probs.cpu().numpy().tolist())
        all_labels.extend(y.cpu().numpy().tolist())
    return total_loss / max(1, n), correct / max(1, n), np.array(all_probs), np.array(all_labels)


# ── Freeze / Unfreeze helpers ──

def freeze_early_layers_generic(model: nn.Module, model_name: str) -> None:
    """
    Freeze approximately the first 50% of layers, model-specific.

    For ResNet-18: freezes conv1, bn1, layer1, layer2 (same as Gnanavel's logic).
    For U-Net and EfficientNet: freezes the first half of all parameters.

    Parameters
    ----------
    model : nn.Module
        Model whose early layers to freeze.
    model_name : str
        One of "resnet18", "unet", "efficientnet".
    """
    if model_name == "resnet18":
        # Same as Gnanavel's freeze logic
        for name, param in model.named_parameters():
            if any(name.startswith(f"backbone.{l}") for l in ["conv1", "bn1", "layer1", "layer2"]):
                param.requires_grad = False
    elif model_name == "unet":
        # Freeze first half of UNet encoder layers
        params = list(model.parameters())
        for p in params[:len(params) // 2]:
            p.requires_grad = False
    elif model_name == "efficientnet":
        # Freeze first half of EfficientNet features
        params = list(model.parameters())
        for p in params[:len(params) // 2]:
            p.requires_grad = False

    trainable = count_trainable(model)
    total = count_parameters(model)
    print(f"  [{model_name}] Trainable: {trainable:,} / {total:,} ({100 * trainable / total:.1f}%)")


def unfreeze_all(model: nn.Module) -> None:
    """Unfreeze all parameters in the model."""
    for p in model.parameters():
        p.requires_grad = True


def train_full_pipeline(model_name: str) -> dict:
    """
    Train one model through both phases (frozen + full fine-tuning).

    Phase 1: Early layers frozen, train classification head (PHASE1_EPOCHS epochs).
    Phase 2: All layers unfrozen, full fine-tuning (PHASE2_EPOCHS epochs).
    Best model saved as best_{model_name}.pth based on validation accuracy.

    Parameters
    ----------
    model_name : str
        One of "resnet18", "unet", "efficientnet".

    Returns
    -------
    dict
        Contains "history" (losses/accuracies), "best_val_acc", "best_epoch".
    """
    print(f"\n{'=' * 75}")
    print(f"  TRAINING: {model_name}")
    print(f"{'=' * 75}")

    model = create_model(model_name).to(device)
    criterion = nn.BCEWithLogitsLoss()
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "phase": []}
    best_val_acc = 0.0
    best_epoch = 0

    # Phase 1: Frozen early layers
    freeze_early_layers_generic(model, model_name)
    opt1 = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=PHASE1_LR)
    sched1 = CosineAnnealingLR(opt1, T_max=PHASE1_EPOCHS, eta_min=1e-6)

    for ep in range(1, PHASE1_EPOCHS + 1):
        tr_l, tr_a = train_one_epoch(model, train_loader, opt1, criterion)
        va_l, va_a, _, _ = eval_one_epoch(model, val_loader, criterion)
        sched1.step()
        history["train_loss"].append(tr_l); history["train_acc"].append(tr_a)
        history["val_loss"].append(va_l); history["val_acc"].append(va_a)
        history["phase"].append(1)
        if va_a > best_val_acc:
            best_val_acc = va_a; best_epoch = ep
            torch.save(model.state_dict(), f"best_{model_name}.pth")
        if ep % 10 == 0 or ep == 1:
            print(f"  P1 {ep:02d}/{PHASE1_EPOCHS} | loss {tr_l:.4f} acc {tr_a:.3f} | val {va_l:.4f} acc {va_a:.3f}")

    # Phase 2: Full fine-tuning
    unfreeze_all(model)
    opt2 = torch.optim.Adam(model.parameters(), lr=PHASE2_LR)
    sched2 = CosineAnnealingLR(opt2, T_max=PHASE2_EPOCHS, eta_min=1e-7)

    for ep in range(1, PHASE2_EPOCHS + 1):
        tr_l, tr_a = train_one_epoch(model, train_loader, opt2, criterion)
        va_l, va_a, _, _ = eval_one_epoch(model, val_loader, criterion)
        sched2.step()
        history["train_loss"].append(tr_l); history["train_acc"].append(tr_a)
        history["val_loss"].append(va_l); history["val_acc"].append(va_a)
        history["phase"].append(2)
        if va_a > best_val_acc:
            best_val_acc = va_a; best_epoch = PHASE1_EPOCHS + ep
            torch.save(model.state_dict(), f"best_{model_name}.pth")
        if ep % 10 == 0 or ep == 1:
            print(f"  P2 {ep:02d}/{PHASE2_EPOCHS} | loss {tr_l:.4f} acc {tr_a:.3f} | val {va_l:.4f} acc {va_a:.3f}")

    print(f"\n  Best val accuracy: {best_val_acc:.4f} (epoch {best_epoch})")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {"history": history, "best_val_acc": best_val_acc, "best_epoch": best_epoch}


# ============================================================
# QUICK TEST — 1 epoch only to verify training loop works
# ============================================================

print("--- Quick Training Test (1 epoch, resnet18) ---")
_test_model = create_model("resnet18").to(device)
_test_crit = nn.BCEWithLogitsLoss()
_test_opt = torch.optim.Adam(_test_model.parameters(), lr=1e-3)
tr_l, tr_a = train_one_epoch(_test_model, train_loader, _test_opt, _test_crit)
va_l, va_a, _, _ = eval_one_epoch(_test_model, val_loader, _test_crit)
print(f"  1 epoch: train_loss={tr_l:.4f}, train_acc={tr_a:.3f}, val_acc={va_a:.3f}")
assert 0 <= tr_a <= 1 and 0 <= va_a <= 1
del _test_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("PASSED: Training loop works")

## 15) Train All Models

In [ ]:
all_results = {}

for model_name in MODEL_NAMES:
    result = train_full_pipeline(model_name)
    all_results[model_name] = result
    print(f"\n{'\u2500' * 40}")
    print(f"  {model_name} COMPLETE \u2014 best val acc: {result['best_val_acc']:.4f}")
    print(f"{'\u2500' * 40}\n")

# Summary
print("=" * 70)
print("  ALL TRAINING COMPLETE")
print("=" * 70)
for name in MODEL_NAMES:
    r = all_results[name]
    print(f"  {name:15s} | best val acc: {r['best_val_acc']:.4f} (epoch {r['best_epoch']})")
    print(f"  {'':15s} | saved: best_{name}.pth")
print("=" * 70)

## 16) Training Curves (All Models)

In [ ]:
# Training curves for all 3 models

colors = {"resnet18": "#1f77b4", "unet": "#ff7f0e", "efficientnet": "#2ca02c"}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name in MODEL_NAMES:
    h = all_results[name]["history"]
    epochs = range(1, len(h["train_loss"]) + 1)
    c = colors[name]

    # Loss
    axes[0].plot(epochs, h["train_loss"], color=c, linestyle="--", alpha=0.5, label=f"{name} train")
    axes[0].plot(epochs, h["val_loss"], color=c, linestyle="-", label=f"{name} val")

    # Accuracy
    axes[1].plot(epochs, h["train_acc"], color=c, linestyle="--", alpha=0.5, label=f"{name} train")
    axes[1].plot(epochs, h["val_acc"], color=c, linestyle="-", label=f"{name} val")

# Phase boundary line
for ax in axes:
    ax.axvline(x=PHASE1_EPOCHS, color="gray", linestyle=":", alpha=0.7, label="Phase boundary")
    ax.legend(fontsize=8)
    ax.set_xlabel("Epoch")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Loss")
axes[0].set_title("Training & Validation Loss — All Models")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Training & Validation Accuracy — All Models")

plt.tight_layout()
plt.show()
print("Training curves plotted for all 3 models.")

## 17) Evaluation â€” Per-Model Metrics

In [ ]:
# Per-model evaluation on test set

results = {}

for name in MODEL_NAMES:
    print(f"\n{'=' * 60}")
    print(f"  EVALUATING: {name}")
    print(f"{'=' * 60}")

    # Load best checkpoint
    model = create_model(name).to(device)
    ckpt_path = f"best_{name}.pth"
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"  Loaded: {ckpt_path}")

    # Evaluate on test set
    criterion = nn.BCEWithLogitsLoss()
    test_loss, test_acc, all_probs, all_labels = eval_one_epoch(model, test_loader, criterion)

    # Compute metrics
    all_preds = (all_probs >= 0.5).astype(float)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="binary")
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.0

    results[name] = {
        "acc": test_acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc,
        "params": count_parameters(model),
        "all_probs": all_probs,
        "all_labels": all_labels,
        "all_preds": all_preds,
    }

    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  AUC-ROC:   {auc:.4f}")

    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay(cm, display_labels=["Clean", "Corrupt"]).plot(ax=ax, cmap="Blues")
    ax.set_title(f"{name} — Confusion Matrix")
    plt.tight_layout()
    plt.show()

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll 3 models evaluated on test set.")

## 18) Per-Corruption-Type Breakdown

In [ ]:
# Per-corruption-type accuracy breakdown for all 3 models

# Load ctype labels from test set .npz files
test_ctypes = []
test_labels_from_npz = []
test_npz_paths = sorted((OUT_DIR / "test").glob("*.npz"))
for p in test_npz_paths:
    d = np.load(p, allow_pickle=True)
    test_ctypes.append(str(d["ctype"]))
    test_labels_from_npz.append(int(d["label"]))

test_ctypes = np.array(test_ctypes)
unique_ctypes = sorted(set(test_ctypes))

print("=" * 80)
print("  PER-CORRUPTION-TYPE ACCURACY BREAKDOWN")
print("=" * 80)

# Header
header = f"{'Corruption Type':<30}"
for name in MODEL_NAMES:
    header += f" {name:>12}"
header += f" {'Count':>8}"
print(header)
print("-" * 80)

for ctype in unique_ctypes:
    mask = test_ctypes == ctype
    count = mask.sum()
    true_labels = np.array(test_labels_from_npz)[mask]
    row = f"{ctype:<30}"

    for name in MODEL_NAMES:
        preds = results[name]["all_preds"][mask]
        acc = (preds == true_labels).mean()
        row += f" {acc:>12.4f}"

    row += f" {count:>8}"
    print(row)

print("-" * 80)
print(f"{'OVERALL':<30}", end="")
for name in MODEL_NAMES:
    print(f" {results[name]['acc']:>12.4f}", end="")
print(f" {len(test_ctypes):>8}")
print("=" * 80)

## 19) Cross-Model Benchmarking Table

In [ ]:
# Cross-model benchmarking table

print("=" * 70)
print("  CROSS-MODEL BENCHMARKING TABLE")
print("=" * 70)
print(f"{'Model':<20} {'Accuracy':>10} {'F1':>8} {'AUC-ROC':>10} {'Params':>12}")
print("-" * 65)
for name in MODEL_NAMES:
    r = results[name]
    print(f"{name:<20} {r['acc']:>10.4f} {r['f1']:>8.4f} {r['auc']:>10.4f} {r['params']:>12,}")
print("=" * 70)

## 20) Inference Time & Resource Measurement

In [ ]:
# Inference time and resource measurement

print("=" * 80)
print("  INFERENCE TIME & RESOURCE MEASUREMENT")
print("=" * 80)

dummy_single = torch.randn(1, 1, 64, 128, 128).to(device)
n_runs = 10

print(f"{'Model':<20} {'Inference (s)':>15} {'GPU Mem (GB)':>14} {'Parameters':>14}")
print("-" * 68)

for name in MODEL_NAMES:
    model = create_model(name).to(device)
    model.load_state_dict(torch.load(f"best_{name}.pth", map_location=device))
    model.eval()

    # Warm-up
    with torch.no_grad():
        _ = model(dummy_single)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    # Timed runs
    t_start = time.time()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy_single)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t_end = time.time()

    avg_time = (t_end - t_start) / n_runs

    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 3)
    else:
        peak_mem = 0.0

    params = count_parameters(model)
    print(f"{name:<20} {avg_time:>15.4f} {peak_mem:>14.2f} {params:>14,}")

    # Store in results
    results[name]["inference_time"] = avg_time
    results[name]["gpu_memory_gb"] = peak_mem

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("=" * 80)

## 21) Single-Scan Inference Demo

In [ ]:
# Single-scan inference demo — all corruption types × all models

# Load one test scan (raw .mhd)
demo_path = test_paths[0]
demo_vol = load_mhd(demo_path)
print(f"Demo scan: {demo_path}")
print(f"  Original shape: {demo_vol.shape}")

# Prepare test cases
rng = np.random.default_rng(42)
test_cases = {}

# 1. Clean
test_cases["Clean"] = preprocess(demo_vol)

# 2. Anatomical slicing (middle, 30%)
sliced, _ = apply_anatomical_slicing(demo_vol, removal_frac=0.30, mode="middle", rng=np.random.default_rng(42))
test_cases["Anat. Slicing (middle, 30%)"] = preprocess(sliced)

# 3. Gaussian noise (std=60)
noisy, _ = apply_gaussian_noise(demo_vol, std=60.0, rng=np.random.default_rng(42))
test_cases["Gaussian Noise (std=60)"] = preprocess(noisy)

# 4. Resolution downsampling (factor=0.4)
degraded, _ = apply_resolution_downsampling(demo_vol, factor=0.4)
test_cases["Resolution Down (0.4)"] = preprocess(degraded)

# 5. Combo_3 (all 3)
combo_params = {
    "anatomical_slicing": {"removal_frac": 0.25, "mode": "scattered"},
    "gaussian_noise": {"std": 60.0},
    "resolution_downsampling": {"factor": 0.4}
}
combo_vol, _ = apply_combo_corruption(
    demo_vol,
    ["anatomical_slicing", "gaussian_noise", "resolution_downsampling"],
    combo_params,
    rng=np.random.default_rng(42)
)
test_cases["Combo_3 (all 3)"] = preprocess(combo_vol)

# Run all models on all test cases
print(f"\n{'Test Case':<35}", end="")
for name in MODEL_NAMES:
    print(f" {name:>14}", end="")
print()
print("-" * 80)

for case_name, vol in test_cases.items():
    tensor = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0).float().to(device)  # (1,1,D,H,W)
    row = f"{case_name:<35}"
    for name in MODEL_NAMES:
        model = create_model(name).to(device)
        model.load_state_dict(torch.load(f"best_{name}.pth", map_location=device))
        model.eval()
        with torch.no_grad():
            logit = model(tensor)
            prob = torch.sigmoid(logit).item()
        row += f" {prob:>14.4f}"
        del model
    print(row)
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("-" * 80)
print("Values shown: P(corrupted) — closer to 0 = clean, closer to 1 = corrupted")

## 22) Resource Summary

In [ ]:
# Resource summary + final verification

total_time = time.time() - NOTEBOOK_START
hours = int(total_time // 3600)
mins = int((total_time % 3600) // 60)
secs = int(total_time % 60)

print("=" * 70)
print("  RESOURCE SUMMARY")
print("=" * 70)
print(f"  Total runtime:    {hours}h {mins}m {secs}s")
print(f"  Device:           {device}")
if torch.cuda.is_available():
    print(f"  GPU:              {torch.cuda.get_device_name(0)}")
    print(f"  GPU memory:       {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB total")
print(f"  Dataset size:     {len(train_ds)} train / {len(val_ds)} val / {len(test_ds)} test")
print(f"  Batch size:       {BATCH_SIZE}")
print(f"  Training epochs:  {PHASE1_EPOCHS} + {PHASE2_EPOCHS} per model")
print()

print("  Per-model summary:")
for name in MODEL_NAMES:
    r = results[name]
    print(f"    {name:15s} | acc={r['acc']:.4f} | F1={r['f1']:.4f} | AUC={r['auc']:.4f} "
          f"| params={r['params']:,} | infer={r['inference_time']:.3f}s | GPU={r['gpu_memory_gb']:.2f}GB")

print()
print("=" * 70)
print("  FINAL BENCHMARKING COMPLETE")
print("=" * 70)
print(f"  Models trained: {len(MODEL_NAMES)}")
print(f"  Corruption types: anatomical_slicing, gaussian_noise, resolution_downsampling + combo")
for name in MODEL_NAMES:
    r = results[name]
    print(f"  {name}: acc={r['acc']:.4f}, F1={r['f1']:.4f}, AUC={r['auc']:.4f}")
print("=" * 70)